# 📽️ PLAYWRIGHT - Data Ingestion

This notebook downloads and processes data from:
1. **Cornell Movie Dialogs Corpus** - 220,579 conversational exchanges from 617 movies
2. **IMSDB (Internet Movie Script Database)** - Full movie scripts with scene structure

The combined dataset will be used to train a beat segmentation model.

---
**Runtime:** ~15-30 minutes

## Setup

In [ ]:
!pip install requests beautifulsoup4 lxml tqdm pandas -q

In [25]:
# Mount Google Drive to persist data between sessions
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [26]:
import os
import ast
import requests
import zipfile
import json
import pickle
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm
import pandas as pd

# Create data directories
DATA_PATH = "/content/drive/MyDrive/playwright_data"
CORNELL_PATH = f"{DATA_PATH}/cornell"
IMSDB_PATH = f"{DATA_PATH}/imsdb"

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(CORNELL_PATH, exist_ok=True)
os.makedirs(IMSDB_PATH, exist_ok=True)

print(f"Data will be saved to: {DATA_PATH}")

Data will be saved to: /content/drive/MyDrive/playwright_data


## 1. Cornell Movie Dialogs Corpus

In [27]:
CORNELL_URL = "http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip"

def download_cornell_corpus():
    """Download and extract Cornell Movie Dialogs Corpus."""
    zip_path = f"{DATA_PATH}/cornell_corpus.zip"
    extract_path = f"{DATA_PATH}/cornell_extracted"

    if os.path.exists(f"{extract_path}/cornell movie-dialogs corpus/movie_lines.txt"):
        print("Cornell corpus exists.")
        return extract_path

    response = requests.get(CORNELL_URL, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(zip_path, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc="Downloading") as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("✅ Cornell corpus downloaded and extracted.")
    return extract_path

cornell_extract_path = download_cornell_corpus()

📥 Downloading Cornell Movie Dialogs Corpus...


Downloading: 100%|██████████| 9.92M/9.92M [00:00<00:00, 11.4MB/s]


📦 Extracting...
✅ Cornell corpus downloaded and extracted.


In [28]:
def parse_cornell_lines(filepath):
    """Parse movie_lines.txt from Cornell corpus."""
    lines = []
    with open(filepath, 'r', encoding='iso-8859-1') as f:
        for line in f:
            parts = line.strip().split(' +++$+++ ')
            if len(parts) == 5:
                lines.append({
                    'line_id': parts[0],
                    'character_id': parts[1],
                    'movie_id': parts[2],
                    'character_name': parts[3],
                    'text': parts[4]
                })
    return lines

def parse_cornell_conversations(filepath):
    """Parse movie_conversations.txt from Cornell corpus."""
    conversations = []
    with open(filepath, 'r', encoding='iso-8859-1') as f:
        for line in f:
            parts = line.strip().split(' +++$+++ ')
            if len(parts) == 4:
                line_ids = ast.literal_eval(parts[3])
                conversations.append({
                    'character1_id': parts[0],
                    'character2_id': parts[1],
                    'movie_id': parts[2],
                    'line_ids': line_ids
                })
    return conversations

def parse_cornell_movies(filepath):
    """Parse movie_titles_metadata.txt from Cornell corpus."""
    movies = []
    with open(filepath, 'r', encoding='iso-8859-1') as f:
        for line in f:
            parts = line.strip().split(' +++$+++ ')
            if len(parts) >= 5:
                movies.append({
                    'movie_id': parts[0],
                    'title': parts[1],
                    'year': parts[2],
                    'imdb_rating': parts[3],
                    'genres': ast.literal_eval(parts[5]) if len(parts) > 5 else []
                })
    return movies

In [29]:
# Parse Cornell data
cornell_base = f"{cornell_extract_path}/cornell movie-dialogs corpus"

lines_data = parse_cornell_lines(f"{cornell_base}/movie_lines.txt")
conversations_data = parse_cornell_conversations(f"{cornell_base}/movie_conversations.txt")
movies_data = parse_cornell_movies(f"{cornell_base}/movie_titles_metadata.txt")

print(f"✅ Loaded {len(lines_data):,} lines")
print(f"✅ Loaded {len(conversations_data):,} conversations")
print(f"✅ Loaded {len(movies_data):,} movies")

📖 Parsing Cornell data...
✅ Loaded 304,446 lines
✅ Loaded 83,097 conversations
✅ Loaded 617 movies


In [30]:
# Convert to DataFrames and save
df_lines = pd.DataFrame(lines_data)
df_movies = pd.DataFrame(movies_data)

df_lines.to_parquet(f"{CORNELL_PATH}/cornell_lines.parquet", index=False)
df_movies.to_parquet(f"{CORNELL_PATH}/cornell_movies.parquet", index=False)

with open(f"{CORNELL_PATH}/cornell_conversations.pkl", 'wb') as f:
    pickle.dump(conversations_data, f)

print(f"Saved Cornell data to {CORNELL_PATH}")
df_lines.head()

💾 Saved Cornell data to /content/drive/MyDrive/playwright_data/cornell


,line_id,character_id,movie_id,character_name,text
0,L1045,u0,m0,BIANCA,They do not!
1,L1044,u2,m0,CAMERON,They do to!
2,L985,u0,m0,BIANCA,I hope so.
3,L984,u2,m0,CAMERON,She okay?
4,L925,u0,m0,BIANCA,Let's go.


## 2. IMSDB Scripts

In [31]:
IMSDB_BASE_URL = "https://imsdb.com"

def get_imsdb_script_list():
    """Get list of all available scripts from IMSDB."""
    all_scripts_url = f"{IMSDB_BASE_URL}/all-scripts.html"

    response = requests.get(all_scripts_url, timeout=30)
    soup = BeautifulSoup(response.text, 'html.parser')

    scripts = []
    for link in soup.find_all('a'):
        href = link.get('href', '')
        if '/Movie Scripts/' in href:
            title = link.text.strip()
            if title:
                script_name = href.split('/')[-1].replace(' Script.html', '')
                scripts.append({
                    'title': title,
                    'url': f"{IMSDB_BASE_URL}{href}",
                    'script_name': script_name
                })

    return scripts

script_list = get_imsdb_script_list()
print(f"{len(script_list)} scripts on IMSDB")

🔍 Fetching IMSDB script list...
✅ Found 1302 scripts on IMSDB


In [32]:
import time

def download_imsdb_script(script_info):
    """Download a single script from IMSDB."""
    try:
        response = requests.get(script_info['url'], timeout=30)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find link to actual script
        script_link = soup.find('a', href=lambda x: x and '/scripts/' in x.lower())
        if not script_link:
            return None

        script_url = f"{IMSDB_BASE_URL}{script_link['href']}"
        script_response = requests.get(script_url, timeout=30)
        script_soup = BeautifulSoup(script_response.text, 'html.parser')

        pre_tag = script_soup.find('pre')
        if pre_tag:
            script_text = pre_tag.get_text()
            return {
                'title': script_info['title'],
                'script_name': script_info['script_name'],
                'text': script_text,
                'url': script_info['url']
            }
    except Exception as e:
        pass

    return None

In [33]:

SAMPLE_SIZE = 50

imsdb_parquet_path = f"{IMSDB_PATH}/imsdb_scripts.parquet"
if os.path.exists(imsdb_parquet_path):
    df_imsdb = pd.read_parquet(imsdb_parquet_path)
    print(f"✅ Loaded {len(df_imsdb)} scripts")
else:
    print(f"📥 Downloading {SAMPLE_SIZE} scripts from IMSDB...")

    downloaded_scripts = []
    for script_info in tqdm(script_list[:SAMPLE_SIZE], desc="Downloading scripts"):
        script_data = download_imsdb_script(script_info)
        if script_data:
            downloaded_scripts.append(script_data)
        time.sleep(0.5)

    print(f"\n✅ Successfully downloaded {len(downloaded_scripts)} scripts")

    df_imsdb = pd.DataFrame(downloaded_scripts)
    df_imsdb.to_parquet(imsdb_parquet_path, index=False)
    print(f" Saved to {imsdb_parquet_path}")

📥 Downloading 50 scripts from IMSDB...
⏳ This may take 10-20 minutes. Be patient!



✅ Successfully downloaded 47 scripts
💾 Saved to /content/drive/MyDrive/playwright_data/imsdb/imsdb_scripts.parquet


In [34]:
print(f"\n IMSDB Scripts Summary:")
print(f"Total scripts: {len(df_imsdb)}")
print(f"Average script length: {df_imsdb['text'].str.len().mean():,.0f} characters")
print(f"\nSample titles:")
for title in df_imsdb['title'].head(10).tolist():
    print(f"  • {title}")


📊 IMSDB Scripts Summary:
Total scripts: 47
Average script length: 206,185 characters

Sample titles:
  • Predator9/10
Master and Commander2/10
White Christmas8/10
Fantastic Beasts: The Crimes of Grindelwald10/10
Legend10/10
  • Master and Commander2/10
White Christmas8/10
Fantastic Beasts: The Crimes of Grindelwald10/10
Legend10/10
  • White Christmas8/10
Fantastic Beasts: The Crimes of Grindelwald10/10
Legend10/10
  • Fantastic Beasts: The Crimes of Grindelwald10/10
Legend10/10
  • Legend10/10
  • 10 Things I Hate About You
  • 12
  • 12 and Holding
  • 12 Monkeys
  • 12 Years a Slave


## 3. Data Summary

In [35]:
print("="*50)
print("DATA INGESTION COMPLETE")
print("="*50)
print(f"\n Data saved to: {DATA_PATH}")
print(f"\n Dataset Statistics:")
print(f"  • Cornell Lines: {len(df_lines):,}")
print(f"  • Cornell Movies: {len(df_movies):,}")
print(f"  • IMSDB Scripts: {len(df_imsdb):,}")
print(f"\n Ready for preprocessing (Notebook 02)")

📊 DATA INGESTION COMPLETE

📁 Data saved to: /content/drive/MyDrive/playwright_data

📈 Dataset Statistics:
  • Cornell Lines: 304,446
  • Cornell Movies: 617
  • IMSDB Scripts: 47

✅ Ready for preprocessing (Notebook 02)


---
## Next Steps

Data has been saved to Google Drive:
- `cornell_lines.parquet` - Individual dialogue lines
- `cornell_movies.parquet` - Movie metadata
- `imsdb_scripts.parquet` - Full movie scripts

**Proceed to `02_data_preprocessing.ipynb`** to:
1. Parse scene boundaries from IMSDB scripts
2. Create beat-level annotations
3. Build training dataset for segmentation model